# Antibiotic Resistance Prediction — Deep Learning
## CNN + MLP on K-mer Frequency Spectra from FASTA Sequences
### FYP — Bacterial Genomics Pipeline (Model 1 of 2)

---

## What This Model Does
Given a bacterial genome (FASTA) and an antibiotic name, predict:
> **Is this bacterium Resistant or Susceptible to this antibiotic?**

---

## Feature Extraction Technique: K-mer Frequency Spectrum
Instead of complex sequence alignment, we use **k-mer counting**:
- Slide a window of size *k* across the genome → count every unique k-mer
- Divide counts by total k-mers → **relative frequency vector** (length = 4^k)
- For k=4: 256 features per genome. For k=5: 1024 features per genome.
- This captures **local sequence composition** without needing a reference genome
- Fast, alignment-free, works on any length genome

```
ATGCATGC... (genome, millions of bp)
   ↓  sliding window k=4
ATGC, TGCA, GCAT, CATG, ...  →  count each of 256 possible 4-mers
   ↓  normalize
[0.003, 0.001, 0.006, ...]   →  256-dim frequency vector per genome
```

---

## Label (Teacher) Selected: `Resistant Phenotype` → Binary
| Original Label | Binary Label | Reason |
|---|---|---|
| Susceptible | **0** | Antibiotic works normally |
| Intermediate | **1** | Borderline — treated as at-risk |
| Resistant | **1** | Antibiotic has failed |

We collapse Intermediate + Resistant → 1 because clinically,
Intermediate means the bacteria is evolving resistance and requires higher doses.

---

## Architecture
```
K-mer freq vector (256-dim)
  → Dense(256, relu) + BN + Dropout(0.3)
  → Dense(128, relu) + BN + Dropout(0.3)
  → [128-dim genome embedding]
                                    ↘
Antibiotic one-hot (N_antibiotics)   Concatenate
  → Dense(64, relu) + Dropout(0.2)  ↗
                                    ↓
                              Dense(64, relu)
                              Dense(32, relu)
                                    ↓
                          resistance_output
                          sigmoid → 0 (Susceptible) or 1 (Resistant)
```

---

## How to Combine This with the Mutation Timeline Model (Model 2)
The best approach for your FYP is a **Gradio web app** (runs directly in Colab):
- User uploads a FASTA file, selects an antibiotic
- **Model 1** (this notebook) → outputs: *Resistant / Susceptible + confidence %*
- **Model 2** (mutation timeline) → outputs: *week-by-week resistance evolution chart*
- Both shown on one screen side by side
- Reason: Gradio needs zero frontend coding, deploys in 2 lines, and produces
  a shareable link — perfect for FYP demo and committee presentations.


## 1. Install and Import

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'biopython', 'imbalanced-learn', '-q'])

import numpy as np
import pandas as pd
import os, glob, random
from collections import Counter
from itertools import product

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

print('TensorFlow :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))

## 2. Configuration
**Only change `CSV_DIR` and `FASTA_BASE_DIR` to match your Google Drive paths.**

In [ ]:
# ── Paths (update these) ────────────────────────────────────────────────
CSV_DIR        = r'C:\Users\HP\OneDrive - Higher Education Commission\Desktop\FYP1\sample_mapped_output'
FASTA_BASE_DIR = r'C:\Users\HP\OneDrive - Higher Education Commission\Desktop\FYP1\sample_fasta_output'
CACHE_DIR      = r'C:\Users\HP\OneDrive - Higher Education Commission\Desktop\FYP1\sample_kmer_cache'

# ── K-mer Configuration ─────────────────────────────────────────────────
K = 4                    # 4 → 256 features | 5 → 1 024 features
MAX_GENOME_BP = 500_000  # first 500 k bp per genome (speeds up counting)
N_WORKERS     = 4        # parallel FASTA readers

# ── Label Configuration ─────────────────────────────────────────────────
LABEL_MAP   = {'Susceptible': 0, 'Intermediate': 1, 'Resistant': 1}
LABEL_NAMES = ['Susceptible', 'Resistant']

# ── Model Hyperparameters ────────────────────────────────────────────────
BATCH_SIZE    = 16
EPOCHS        = 80
LEARNING_RATE = 0.0005
DROPOUT       = 0.35
N_FOLDS       = 5

import os
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'K-mer size     : {K}  →  {4**K} features per genome')
print(f'Max genome bp  : {MAX_GENOME_BP:,}')
print(f'Label mapping  : {LABEL_MAP}')
print(f'Cross-val folds: {N_FOLDS}')
print(f'Workers        : {N_WORKERS}')
print(f'Cache dir      : {CACHE_DIR}')


## 3. Build K-mer Alphabet
Pre-generate all possible k-mers for k=4 (AAAA … TTTT) so the feature vector
always has the same 256 dimensions regardless of which k-mers appear in a genome.

In [ ]:
NUCLEOTIDES = ['A', 'T', 'C', 'G']

# All possible k-mers — fixed alphabetical order
ALL_KMERS   = [''.join(p) for p in product(NUCLEOTIDES, repeat=K)]
KMER_TO_IDX = {km: i for i, km in enumerate(ALL_KMERS)}
N_KMERS     = len(ALL_KMERS)   # 4^K

print(f'Total {K}-mers: {N_KMERS}')
print(f'First 8: {ALL_KMERS[:8]}')
print(f'Last  8: {ALL_KMERS[-8:]}')

## 4. Data Loading

### What Each Function Does
| Function | Purpose |
|---|---|
| `resolve_path()` | Converts your Windows `fasta_path` to a Colab-compatible path |
| `kmer_freq_vector()` | Reads FASTA (skips header line 1), counts k-mers, returns 256-dim vector |
| `load_dataset()` | Loads all CSVs, links each row to its FASTA, builds the feature matrix |


In [ ]:
import concurrent.futures
from collections import Counter

# ── Non-ATCG strip table (built once at module load) ─────────────────────
_STRIP = str.maketrans('', '', ''.join(
    c for c in map(chr, range(256)) if c not in 'ATCG'
))

def _read_fasta_seq(path, max_bp=MAX_GENOME_BP):
    """Read multi-contig FASTA → upper-case ATCG string, capped at max_bp."""
    with open(path, 'r') as f:
        seq = ''.join(
            line.strip() for line in f if not line.startswith('>')
        ).upper().translate(_STRIP)
    return seq[:max_bp] if max_bp else seq


def kmer_freq_vector(fasta_path, k=K, max_bp=MAX_GENOME_BP):
    """
    K-mer frequency vector for one FASTA file.
    Uses Counter (C hash-map) — 5-10× faster than dict.get in a Python loop.
    Returns np.array (4^K,) or None on failure.
    """
    cache_file = os.path.join(CACHE_DIR, os.path.basename(fasta_path) + f'.k{k}.npy')
    if os.path.exists(cache_file):
        return np.load(cache_file)

    if not os.path.exists(fasta_path):
        return None

    seq = _read_fasta_seq(fasta_path, max_bp)
    if len(seq) < k + 10:
        return None

    counter = Counter(seq[i:i+k] for i in range(len(seq) - k + 1))
    counts  = np.array([counter.get(km, 0) for km in ALL_KMERS], dtype=np.float32)
    total   = counts.sum()
    if total == 0:
        return None
    vec = counts / total
    np.save(cache_file, vec)   # cache for future runs
    return vec


def _worker(path):
    return path, kmer_freq_vector(path)


def resolve_path(windows_path, fasta_base_dir):
    norm  = windows_path.replace('\\', '/')
    parts = norm.split('/')
    try:
        idx      = next(i for i, p in enumerate(parts) if p == 'fasta_output')
        relative = '/'.join(parts[idx + 1:])
    except StopIteration:
        relative = '/'.join(parts[-2:])
    return os.path.join(fasta_base_dir, relative)


def load_dataset(csv_dir, fasta_base_dir, n_workers=N_WORKERS):
    """
    Load all mapped CSVs and compute k-mer features.
    Parallel FASTA reading via ThreadPoolExecutor.
    Disk cache: processed genomes skip re-computation on re-runs.
    """
    csv_files = glob.glob(os.path.join(csv_dir, '*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No CSV files in {csv_dir}')
    print(f'Found {len(csv_files)} CSV file(s) — loading...')

    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    df = df.dropna(subset=['Genome ID', 'Antibiotic', 'Resistant Phenotype', 'fasta_path'])
    df = df[df['Resistant Phenotype'].isin(LABEL_MAP)].copy()
    df['label']      = df['Resistant Phenotype'].map(LABEL_MAP)
    df['colab_path'] = df['fasta_path'].apply(
        lambda p: resolve_path(str(p), fasta_base_dir)
    )

    ab_list    = sorted(df['Antibiotic'].unique())
    ab_to_idx  = {a: i for i, a in enumerate(ab_list)}
    N_AB       = len(ab_list)
    ab_onehots = np.eye(N_AB, dtype=np.float32)  # vectorised one-hots

    unique_paths = df['colab_path'].unique().tolist()
    print(f'Processing {len(unique_paths)} unique genomes (workers={n_workers})...')

    kmer_cache = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as exe:
        futs = {exe.submit(_worker, p): p for p in unique_paths}
        done = 0
        step = max(1, len(unique_paths) // 10)
        for fut in concurrent.futures.as_completed(futs):
            path, vec = fut.result()
            kmer_cache[path] = vec
            done += 1
            if done % step == 0 or done == len(unique_paths):
                print(f'  {done}/{len(unique_paths)} genomes done')

    mask  = df['colab_path'].map(lambda p: kmer_cache.get(p) is not None)
    valid = df[mask].reset_index(drop=True)

    X_kmer = np.stack(
        [kmer_cache[p] for p in valid['colab_path']]
    ).astype(np.float32)
    X_ab   = ab_onehots[valid['Antibiotic'].map(ab_to_idx).values]
    y      = valid['label'].values.astype(np.int32)
    meta_df = valid[['Genome ID', 'Antibiotic', 'Resistant Phenotype', 'label']].copy()
    meta_df.columns = ['genome_id', 'antibiotic', 'phenotype', 'label']

    print(f'\nLoaded {len(y)} samples from {len(kmer_cache)} unique genomes')
    print(f'Antibiotics : {N_AB}  |  K-mer features : {X_kmer.shape[1]}')
    print(f'Class dist  : 0={int((y==0).sum())}  1={int((y==1).sum())}')
    return X_kmer, X_ab, y, meta_df, ab_list


X_kmer, X_ab, y, meta_df, ab_list = load_dataset(CSV_DIR, FASTA_BASE_DIR)
N_ANTIBIOTICS = len(ab_list)
print(f'\nX_kmer : {X_kmer.shape}  |  X_ab : {X_ab.shape}  |  y : {y.shape}')


## 5. Exploratory Data Analysis
Quick look at the data distribution before training.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Dataset Overview', fontsize=14, fontweight='bold')

# Class distribution
counts = pd.Series(y).map({0:'Susceptible', 1:'Resistant/I'}).value_counts()
axes[0].bar(counts.index, counts.values,
            color=['#4CAF50','#F44336'], edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# Antibiotic distribution
ab_counts = meta_df['antibiotic'].value_counts().head(15)
axes[1].barh(ab_counts.index[::-1], ab_counts.values[::-1], color='#2196F3')
axes[1].set_title('Samples per Antibiotic (top 15)')
axes[1].set_xlabel('Count')

# Phenotype per antibiotic (top 10)
top_ab = meta_df['antibiotic'].value_counts().head(10).index
pt_tab = meta_df[meta_df['antibiotic'].isin(top_ab)]\
           .groupby(['antibiotic','phenotype']).size().unstack(fill_value=0)
pt_tab.plot(kind='bar', ax=axes[2], color=['#F44336','#FF9800','#4CAF50'],
            edgecolor='black')
axes[2].set_title('Phenotype Breakdown (top 10 antibiotics)')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\nMeta data sample:')
print(meta_df.head(10).to_string(index=False))

## 6. Preprocessing

Three steps — intentionally kept minimal:
1. **StandardScaler on k-mer features** — scales each of the 256 features to mean=0, std=1
   (important because GC-rich genomes have very different raw frequencies)
2. **SMOTE oversampling** — synthetic minority oversampling to fix class imbalance
   (your dataset has more Susceptible than Resistant)
3. **Train/test split** — 80/20, stratified on label

In [ ]:
# Combine k-mer + antibiotic features into one matrix
X_full = np.concatenate([X_kmer, X_ab], axis=1)
print(f'Combined feature matrix: {X_full.shape}')
print(f'  K-mer features   : {X_kmer.shape[1]}')
print(f'  Antibiotic feats : {X_ab.shape[1]}')

# Train / test split — BEFORE scaling to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y
)

# Scale only the k-mer portion (antibiotic one-hot doesn't need scaling)
n_kmer = X_kmer.shape[1]
scaler = StandardScaler()
X_train[:, :n_kmer] = scaler.fit_transform(X_train[:, :n_kmer])
X_test [:, :n_kmer] = scaler.transform    (X_test [:, :n_kmer])

# SMOTE — only on training set
print(f'\nBefore SMOTE: {Counter(y_train)}')
try:
    smote = SMOTE(random_state=42, k_neighbors=min(5, Counter(y_train).get(1,1)-1))
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print(f'After  SMOTE: {Counter(y_train_bal)}')
except Exception as e:
    print(f'SMOTE skipped ({e}) — using original imbalanced data')
    X_train_bal, y_train_bal = X_train, y_train

print(f'\nTrain set: {X_train_bal.shape[0]} samples')
print(f'Test  set: {X_test.shape[0]} samples')

## 7. Model Architecture

Two-branch MLP that processes k-mer features and antibiotic condition separately,
then merges them for the final resistance prediction.

**Why MLP (not CNN) here?**
The k-mer frequency vector has **no spatial order** — `AAAA` at index 0 and `TTTT`
at index 255 have no positional relationship. CNNs are designed for ordered sequences.
An MLP is the correct architecture for a fixed-length frequency feature vector.

For the mutation timeline model (Model 2) we use CNN+LSTM because there the
*position* of each k-mer in the sliding window carries meaning.

In [ ]:
def build_resistance_model(n_kmer_features, n_antibiotics, dropout=DROPOUT):
    """
    Two-branch MLP for antibiotic resistance prediction.

    Branch A: k-mer frequency features → deep MLP
    Branch B: antibiotic one-hot       → small embedding
    Merged  : shared dense layers → binary sigmoid output
    """
    # ── Branch A: Genomic k-mer features ───────────────────────────────
    kmer_input = layers.Input(shape=(n_kmer_features,), name='kmer_input')

    a = layers.Dense(256, name='kmer_d1')(kmer_input)
    a = layers.BatchNormalization()(a)
    a = layers.Activation('relu')(a)
    a = layers.Dropout(dropout)(a)

    a = layers.Dense(128, name='kmer_d2')(a)
    a = layers.BatchNormalization()(a)
    a = layers.Activation('relu')(a)
    a = layers.Dropout(dropout)(a)

    a = layers.Dense(64, activation='relu', name='kmer_d3')(a)
    # → (batch, 64)  genome embedding

    # ── Branch B: Antibiotic condition ─────────────────────────────────
    ab_input = layers.Input(shape=(n_antibiotics,), name='antibiotic_input')

    b = layers.Dense(64, activation='relu', name='ab_d1')(ab_input)
    b = layers.Dropout(0.2)(b)
    b = layers.Dense(32, activation='relu', name='ab_d2')(b)
    # → (batch, 32)  antibiotic embedding

    # ── Merge ───────────────────────────────────────────────────────────
    merged = layers.Concatenate(name='merge')([a, b])   # (batch, 96)

    z = layers.Dense(64, activation='relu', name='merged_d1')(merged)
    z = layers.Dropout(dropout)(z)
    z = layers.Dense(32, activation='relu', name='merged_d2')(z)
    z = layers.Dropout(0.2)(z)

    # ── Output: resistance probability ─────────────────────────────────
    output = layers.Dense(1, activation='sigmoid', name='resistance_output')(z)

    model = models.Model(
        inputs  = [kmer_input, ab_input],
        outputs = output,
        name    = 'ResistancePredictor'
    )
    return model


model = build_resistance_model(n_kmer + N_ANTIBIOTICS - N_ANTIBIOTICS, N_ANTIBIOTICS)
# Note: kmer_input gets first n_kmer columns, ab_input gets last N_ANTIBIOTICS columns

# Class weights — compensate for imbalance even after SMOTE
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_train_bal), y=y_train_bal)
class_weight = {0: cw[0], 1: cw[1]}
print(f'Class weights: {class_weight}')

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

model.summary()

## 8. Training

In [ ]:
# Split k-mer and antibiotic features back apart for two-branch model
def split_inputs(X, n_kmer):
    return X[:, :n_kmer], X[:, n_kmer:]

Xk_tr, Xa_tr = split_inputs(X_train_bal, n_kmer)
Xk_te, Xa_te = split_inputs(X_test,      n_kmer)

cb_early = callbacks.EarlyStopping(
    monitor='val_auc', mode='max',
    patience=15, restore_best_weights=True, verbose=1
)
cb_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=7,
    min_lr=1e-6, verbose=1
)
cb_ckpt = callbacks.ModelCheckpoint(
    'best_resistance_model.h5',
    monitor='val_auc', mode='max',
    save_best_only=True, verbose=0
)

print('Training...')
history = model.fit(
    x               = [Xk_tr, Xa_tr],
    y               = y_train_bal,
    validation_split= 0.2,
    batch_size      = BATCH_SIZE,
    epochs          = EPOCHS,
    class_weight    = class_weight,
    callbacks       = [cb_early, cb_lr, cb_ckpt],
    verbose         = 1
)
print('Done.')

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Resistance Predictor — Training History', fontsize=14, fontweight='bold')

panels = [
    ('loss',      'val_loss',      'Loss',              axes[0,0]),
    ('accuracy',  'val_accuracy',  'Accuracy',          axes[0,1]),
    ('auc',       'val_auc',       'ROC-AUC ← Key',     axes[1,0]),
    ('precision', 'val_precision', 'Precision / Recall', axes[1,1]),
]

for tr_key, va_key, title, ax in panels:
    if tr_key in history.history:
        ax.plot(history.history[tr_key], label='Train', linewidth=2)
    if va_key in history.history:
        ax.plot(history.history[va_key], label='Val', linewidth=2, linestyle='--')
    # Add recall to the last panel
    if title == 'Precision / Recall' and 'recall' in history.history:
        ax.plot(history.history['recall'],     label='Train Recall', linewidth=1.5, linestyle=':')
        ax.plot(history.history['val_recall'], label='Val Recall',   linewidth=1.5, linestyle='-.')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Evaluation on Test Set

In [ ]:
y_prob = model.predict([Xk_te, Xa_te], verbose=0).squeeze()
y_pred = (y_prob >= 0.5).astype(int)

print('=' * 50)
print('CLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(y_test, y_pred,
                             target_names=LABEL_NAMES, digits=4))

try:
    auc_score = roc_auc_score(y_test, y_prob)
    print(f'ROC-AUC : {auc_score:.4f}')
except Exception as e:
    print(f'ROC-AUC : N/A ({e})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Test Set Evaluation', fontsize=14, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            annot_kws={'size': 14}, ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalised confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            annot_kws={'size': 14}, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalised)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

# ROC curve
try:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val     = roc_auc_score(y_test, y_prob)
    axes[2].plot(fpr, tpr, color='crimson', linewidth=2,
                 label=f'ROC (AUC = {auc_val:.3f})')
    axes[2].plot([0,1],[0,1],'k--', linewidth=1, label='Random')
    axes[2].set_xlabel('False Positive Rate')
    axes[2].set_ylabel('True Positive Rate')
    axes[2].set_title('ROC Curve')
    axes[2].legend(fontsize=10)
    axes[2].grid(True, alpha=0.3)
except Exception as e:
    axes[2].text(0.5, 0.5, f'ROC N/A\n{e}', ha='center', va='center')

plt.tight_layout()
plt.show()

## 11. Per-Antibiotic Accuracy Breakdown
Shows how well the model performs for each antibiotic separately.
Useful for identifying which drugs the model finds hardest to predict.

In [ ]:
# Predict on full dataset for per-antibiotic analysis
Xk_all = X_full[:, :n_kmer].copy()
Xa_all = X_full[:, n_kmer:].copy()
Xk_all[:, :n_kmer] = scaler.transform(Xk_all)   # scale

y_prob_all = model.predict([Xk_all, Xa_all], verbose=0).squeeze()
y_pred_all = (y_prob_all >= 0.5).astype(int)

meta_df['predicted']    = y_pred_all
meta_df['probability']  = y_prob_all
meta_df['correct']      = (meta_df['predicted'] == meta_df['label']).astype(int)

per_ab = (meta_df.groupby('antibiotic')
              .agg(
                  samples =('correct','count'),
                  accuracy=('correct','mean'),
                  avg_prob=('probability','mean')
              )
              .sort_values('accuracy', ascending=False))

print('Per-Antibiotic Results:')
print(per_ab.to_string())

fig, ax = plt.subplots(figsize=(14, 6))
colors  = ['#4CAF50' if v >= 0.7 else '#FF9800' if v >= 0.5 else '#F44336'
            for v in per_ab['accuracy']]
ax.bar(per_ab.index, per_ab['accuracy'] * 100, color=colors, edgecolor='black')
ax.axhline(70, color='green',  linestyle='--', linewidth=1.2, label='70% threshold')
ax.axhline(50, color='orange', linestyle='--', linewidth=1.2, label='50% threshold')
ax.set_title('Per-Antibiotic Prediction Accuracy', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_xlabel('Antibiotic')
ax.tick_params(axis='x', rotation=60)
ax.set_ylim(0, 110)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
for i, (acc, cnt) in enumerate(zip(per_ab['accuracy'], per_ab['samples'])):
    ax.text(i, acc*100 + 2, f'n={cnt}', ha='center', fontsize=7)
plt.tight_layout()
plt.show()

## 12. K-fold Cross Validation
Because the dataset is small, k-fold gives a more reliable estimate of
true performance than a single train/test split.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

print(f'Running {N_FOLDS}-fold stratified cross-validation...')

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y)):
    Xk_tr_f = X_full[tr_idx, :n_kmer].copy()
    Xa_tr_f = X_full[tr_idx, n_kmer:]
    Xk_va_f = X_full[va_idx, :n_kmer].copy()
    Xa_va_f = X_full[va_idx, n_kmer:]
    y_tr_f  = y[tr_idx]
    y_va_f  = y[va_idx]

    sc_f = StandardScaler()
    Xk_tr_f = sc_f.fit_transform(Xk_tr_f)
    Xk_va_f = sc_f.transform(Xk_va_f)

    # SMOTE on fold training data
    try:
        sm    = SMOTE(random_state=42, k_neighbors=min(3, Counter(y_tr_f).get(1,1)-1))
        Xk_smote, y_sm = sm.fit_resample(Xk_tr_f, y_tr_f)
        Xa_smote = np.tile(Xa_tr_f.mean(axis=0), (len(y_sm),1))  # approximate
    except:
        Xk_smote, Xa_smote, y_sm = Xk_tr_f, Xa_tr_f, y_tr_f

    fold_model = build_resistance_model(n_kmer, N_ANTIBIOTICS)
    fold_model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    fold_model.fit(
        [Xk_smote, Xa_smote], y_sm,
        validation_data=([Xk_va_f, Xa_va_f], y_va_f),
        epochs=50, batch_size=BATCH_SIZE,
        callbacks=[callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                           patience=10, restore_best_weights=True)],
        verbose=0
    )

    y_prob_f = fold_model.predict([Xk_va_f, Xa_va_f], verbose=0).squeeze()
    y_pred_f = (y_prob_f >= 0.5).astype(int)
    acc_f    = (y_pred_f == y_va_f).mean()
    try:    auc_f = roc_auc_score(y_va_f, y_prob_f)
    except: auc_f = float('nan')

    fold_results.append({'fold': fold+1, 'accuracy': acc_f, 'auc': auc_f})
    print(f'  Fold {fold+1}/{N_FOLDS} — Accuracy: {acc_f:.4f} | AUC: {auc_f:.4f}')

fr_df = pd.DataFrame(fold_results)
print(f'\nMean Accuracy : {fr_df["accuracy"].mean():.4f} ± {fr_df["accuracy"].std():.4f}')
print(f'Mean AUC      : {fr_df["auc"].mean():.4f} ± {fr_df["auc"].std():.4f}')

## 13. Resistance Prediction Function

In [ ]:
def predict_resistance(fasta_file, antibiotic, threshold=0.5):
    """
    Predict whether a bacterium is resistant to a given antibiotic.

    Args:
        fasta_file : path to .fasta file (header on line 1, sequence from line 2)
        antibiotic : antibiotic name string (must match training antibiotics)
        threshold  : probability cutoff (default 0.5)

    Returns:
        dict with keys: prediction, probability, confidence, label
    """
    # K-mer features from FASTA
    vec = kmer_freq_vector(fasta_file)
    if vec is None:
        return {'error': f'Could not read FASTA: {fasta_file}'}

    vec_scaled = scaler.transform(vec.reshape(1, -1))

    # Antibiotic one-hot
    if antibiotic not in ab_list:
        print(f'Warning: "{antibiotic}" not in training antibiotics.')
        print(f'Known: {ab_list}')
        ab_vec = np.zeros((1, N_ANTIBIOTICS), dtype=np.float32)
    else:
        ab_vec    = np.zeros((1, N_ANTIBIOTICS), dtype=np.float32)
        ab_vec[0, ab_list.index(antibiotic)] = 1.0

    prob  = float(model.predict([vec_scaled, ab_vec], verbose=0).squeeze())
    label = 'Resistant' if prob >= threshold else 'Susceptible'
    conf  = prob if prob >= threshold else 1 - prob

    return {
        'antibiotic' : antibiotic,
        'prediction' : label,
        'probability': round(prob, 4),
        'confidence' : f'{conf*100:.1f}%',
        'genome'     : os.path.basename(fasta_file)
    }


# ── Example usage ───────────────────────────────────────────────────────
# result = predict_resistance(
#     '/content/drive/MyDrive/.../106654.496.fasta',
#     antibiotic='ciprofloxacin'
# )
# print(result)
print('predict_resistance() is ready.')
print(f'Supported antibiotics ({N_ANTIBIOTICS}):', ab_list)

## 14. Batch Prediction on a Folder of FASTA Files
Scan all FASTA files in a folder and predict resistance for a given antibiotic.

In [ ]:
def batch_predict_folder(fasta_folder, antibiotic):
    """
    Run predict_resistance() on every .fasta file in a folder.
    Returns a DataFrame sorted by probability (highest resistance first).
    """
    fasta_files = glob.glob(os.path.join(fasta_folder, '*.fasta'))
    print(f'Found {len(fasta_files)} FASTA files in {fasta_folder}')

    results = []
    for fasta in fasta_files:
        r = predict_resistance(fasta, antibiotic)
        if 'error' not in r:
            results.append(r)

    df = pd.DataFrame(results).sort_values('probability', ascending=False)
    return df


# ── Example ──────────────────────────────────────────────────────
# result_df = batch_predict_folder(
#     '/content/drive/MyDrive/Fyp_Data/fasta_output/taxon_106654_Acinetobacter_nosocomialis',
#     antibiotic='ciprofloxacin'
# )
# print(result_df.to_string(index=False))
print('batch_predict_folder() is ready.')

## 15. Save Model and Artifacts

In [ ]:
import pickle

SAVE_DIR = '/content/drive/MyDrive/Fyp_Data/resistance_model'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, 'resistance_predictor.h5'))

artifacts = {
    'scaler'      : scaler,
    'ab_list'     : ab_list,
    'N_ANTIBIOTICS': N_ANTIBIOTICS,
    'n_kmer'      : n_kmer,
    'K'           : K,
    'ALL_KMERS'   : ALL_KMERS,
    'KMER_TO_IDX' : KMER_TO_IDX,
    'NUCLEOTIDES' : NUCLEOTIDES,
    'LABEL_MAP'   : LABEL_MAP,
    'LABEL_NAMES' : LABEL_NAMES,
    'MAX_GENOME_BP': MAX_GENOME_BP
}
with open(os.path.join(SAVE_DIR, 'artifacts.pkl'), 'wb') as f:
    pickle.dump(artifacts, f)

print('Saved:')
print(f'  {SAVE_DIR}/resistance_predictor.h5')
print(f'  {SAVE_DIR}/artifacts.pkl')

## 16. Load Model for Future Sessions

In [ ]:
def load_resistance_model(save_dir):
    """
    Reload the model and all preprocessing artifacts in a new Colab session.
    After calling this, predict_resistance() works without retraining.
    """
    import pickle
    loaded_model = keras.models.load_model(
        os.path.join(save_dir, 'resistance_predictor.h5')
    )
    with open(os.path.join(save_dir, 'artifacts.pkl'), 'rb') as f:
        art = pickle.load(f)

    # Restore globals
    globals()['model']        = loaded_model
    globals()['scaler']       = art['scaler']
    globals()['ab_list']      = art['ab_list']
    globals()['N_ANTIBIOTICS']= art['N_ANTIBIOTICS']
    globals()['n_kmer']       = art['n_kmer']
    globals()['K']            = art['K']
    globals()['ALL_KMERS']    = art['ALL_KMERS']
    globals()['KMER_TO_IDX']  = art['KMER_TO_IDX']
    globals()['NUCLEOTIDES']  = art['NUCLEOTIDES']
    globals()['MAX_GENOME_BP']= art['MAX_GENOME_BP']

    print('Model and artifacts loaded successfully.')
    return loaded_model, art


# Uncomment to use in a new session:
# model, art = load_resistance_model('/content/drive/MyDrive/Fyp_Data/resistance_model')
print('load_resistance_model() defined.')

## 17. How to Combine With the Mutation Timeline Model (FYP Integration)

### Recommended Approach: Gradio Web App

**Why Gradio over other options:**

| Option | Effort | Demo Quality | Works in Colab? |
|---|---|---|---|
| **Gradio** | ⭐ Very low | ⭐⭐⭐ Professional | ✅ Yes, with shareable link |
| Streamlit | Medium | ⭐⭐⭐ Professional | ⚠ Needs ngrok tunnel |
| Flask/Django | High | ⭐⭐⭐ Professional | ❌ No |
| Jupyter widgets | Low | ⭐ Basic | ✅ Yes |

### Integration Architecture
```
User input: [FASTA file] + [Antibiotic dropdown]
                ↓
       ┌────────────────────┐
       │   Model 1 (this)   │ → Is it resistant? (Yes/No + 87% confidence)
       └────────────────────┘
                ↓
       ┌────────────────────┐
       │   Model 2 (timeline)│ → Week-by-week resistance evolution chart
       └────────────────────┘
                ↓
       Combined Gradio output:
         - Resistance badge (green/red)
         - Confidence percentage
         - Timeline line chart
         - Top mutation sites table
```

### Starter Gradio Code (run this in a separate cell after both models are loaded)

In [ ]:
# ── Gradio Integration Starter ─────────────────────────────────────────
# Install: pip install gradio
# This cell shows the structure — fill in model_2_timeline() with
# your mutation timeline model from the other notebook.

# import gradio as gr
# import tempfile
#
# def combined_predict(fasta_file_obj, antibiotic):
#     # Save uploaded file
#     with tempfile.NamedTemporaryFile(suffix='.fasta', delete=False) as tmp:
#         tmp.write(fasta_file_obj)
#         tmp_path = tmp.name
#
#     # ── Model 1: Resistance Prediction ────────────────────────────────
#     res = predict_resistance(tmp_path, antibiotic)
#     resistance_text = (
#         f"Prediction : {res['prediction']}\n"
#         f"Confidence : {res['confidence']}\n"
#         f"Raw prob   : {res['probability']}"
#     )
#
#     # ── Model 2: Timeline (import from mutation notebook) ─────────────
#     # tdf, _, fail_wk = predict_fasta_timeline(tmp_path, antibiotic)
#     # fig = plot_timeline(tdf, antibiotic, fail_wk)  # your plot function
#
#     return resistance_text  # , fig  (add timeline figure when ready)
#
# demo = gr.Interface(
#     fn      = combined_predict,
#     inputs  = [
#         gr.File(label='Upload FASTA file',  file_types=['.fasta']),
#         gr.Dropdown(label='Antibiotic', choices=ab_list)
#     ],
#     outputs = [
#         gr.Textbox(label='Resistance Prediction'),
#         # gr.Plot(label='Mutation Timeline')   # uncomment when Model 2 ready
#     ],
#     title       = 'Bacterial AMR Predictor — FYP Demo',
#     description = 'Upload a bacterial genome FASTA and select an antibiotic.'
# )
# demo.launch(share=True)   # share=True gives a public link

print('Gradio integration template ready.')
print('Uncomment when both models are trained.')

## Summary

### Features Used From Your CSV
| Column | Role in This Model |
|---|---|
| `fasta_path` | Resolves to genome sequence → k-mer frequency vector (256 features) |
| `Antibiotic` | One-hot encoded condition (31 features in this sample CSV) |
| `Resistant Phenotype` | **Label (teacher signal)** — Susceptible=0, Intermediate+Resistant=1 |
| `Measurement Value` | Not used in features (too many NaN), but informs label quality |
| `Genome ID` | Used for caching — same genome reads FASTA only once |

### Why K-mer Frequency? (Feature Extraction Justification)
- **No reference genome needed** — works on any new genome out of the box
- **Fixed-length vector** — every genome, regardless of length, gives 256 numbers
- **Biologically meaningful** — codon composition, GC content, repeat patterns
  are all captured in 4-mer frequencies
- **Proven** — k-mer spectra are the standard feature for AMR prediction in
  publications (Patric, ResFinder, Kover all use them)

### Notebooks in Your FYP Pipeline
| Notebook | Input | Output |
|---|---|---|
| **This one** | FASTA + Antibiotic | Resistant / Susceptible + confidence |
| Mutation Timeline | FASTA + Antibiotic | Week-by-week resistance % + nucleotide changes |
| Gradio App (next) | Both saved models | Combined interactive demo |
